# 02 Support Vector Machine (SVM)

Notebook to train and evaluate the SVM model with word embeddings as the feature extraction method

## 1 Import libraries

In [32]:
import pandas as pd
import nltk
import numpy as np
import spacy
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    roc_auc_score, 
    classification_report
)

In [10]:
# Download the sentence tokenizer model
nltk.download('punkt')
nltk.download('stopwords')

from nltk.tokenize import sent_tokenize

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## 2 Load data

In [ ]:
# Load the training data
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
test_labels = pd.read_csv('data/test_labels.csv')
categories = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

## 3 Load spaCy model

In [3]:
# !python -m spacy download en_core_web_md
nlp = spacy.load("en_core_web_md", disable=["tagger", "parser", "ner"])

## 4 Prepare the data

In [16]:
def get_embedding(text):
    """Function to convert text to an average word vector"""
    doc = nlp(text)
    if not doc.has_vector:  # Check if doc has vectors
        return np.zeros(doc.vector.shape[0])
    return doc.vector

In [20]:
# Calculating embeddings and preparing data for SVM
X_train = np.array([get_embedding(str(text)) for text in train['comment_text']])
y_train = train[categories].values

C:\Users\ASUS\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\spacy\pipeline\lemmatizer.py:187: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


## 5 Build and Train the Model

In [21]:
svm_model = OneVsRestClassifier(LinearSVC(class_weight='balanced', max_iter=2000))
svm_model.fit(X_train, y_train)

,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",LinearSVC(cla...max_iter=2000)
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",None
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower 

## 6 Predictions

In [24]:
test_combined = pd.merge(test, test_labels, on='id')

In [ ]:
# If any label is -1, the whole row was not used for scoring
test_scored = test_combined[test_combined['toxic'] != -1].copy()

In [27]:
print(f"Original test rows: {len(test_combined)}")
print(f"Scored test rows after filtering: {len(test_scored)}")

Original test rows: 153164
Scored test rows after filtering: 63978


In [28]:
# Get Embeddings for the filtered Test Set
X_test = np.array([get_embedding(str(text)) for text in test_scored['comment_text']])
y_test = test_scored[categories].values

C:\Users\ASUS\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\spacy\pipeline\lemmatizer.py:187: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


In [29]:
y_pred = svm_model.predict(X_test)
y_score = svm_model.decision_function(X_test)

## 7 Metrics

In [30]:
metrics_list = []

In [33]:
for i, category in enumerate(categories):
    # Calculate metrics for this specific category
    acc = accuracy_score(y_test[:, i], y_pred[:, i])
    prec = precision_score(y_test[:, i], y_pred[:, i], zero_division=0)
    rec = recall_score(y_test[:, i], y_pred[:, i], zero_division=0)
    f1 = f1_score(y_test[:, i], y_pred[:, i], zero_division=0)
    auc = roc_auc_score(y_test[:, i], y_score[:, i])
    
    metrics_list.append({
        'Category': category,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'AUC': auc
    })

In [35]:
results_df = pd.DataFrame(metrics_list)
results_df.round(4)

,Category,Accuracy,Precision,Recall,F1-Score,AUC
0,toxic,0.8489,0.3697,0.8322,0.5119,0.9187
1,severe_toxic,0.9098,0.0553,0.9155,0.1043,0.9667
2,obscene,0.8843,0.3097,0.8187,0.4494,0.9323
3,threat,0.9383,0.0440,0.8531,0.0836,0.9568
4,insult,0.8875,0.2994,0.8203,0.4386,0.9333
5,identity_hate,0.8941,0.0847,0.8680,0.1543,0.9449


In [36]:
# Calculate Global (Macro) Metrics
# Macro average calculates the metric for each label and finds their unweighted mean.
macro_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
macro_prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
macro_rec = recall_score(y_test, y_pred, average='macro', zero_division=0)
macro_auc = roc_auc_score(y_test, y_score, average='macro')

In [37]:
subset_accuracy = accuracy_score(y_test, y_pred)

In [38]:
print("--- Summary Global Metrics (Macro Average) ---")
print(f"Exact Match Accuracy: {subset_accuracy:.4f}")
print(f"Macro Precision:      {macro_prec:.4f}")
print(f"Macro Recall:         {macro_rec:.4f}")
print(f"Macro F1-Score:       {macro_f1:.4f}")
print(f"Macro AUC:            {macro_auc:.4f}")

--- Summary Global Metrics (Macro Average) ---
Exact Match Accuracy: 0.7453
Macro Precision:      0.1938
Macro Recall:         0.8513
Macro F1-Score:       0.2904
Macro AUC:            0.9421


End of notebook